# Assignment 2: Feedforward Neural Network (CIFAR-10)
**Aim:** Implement a feedforward neural network with Keras and TensorFlow.

a. Import packages  b. Load data  c. Define network  d. Train with SGD  e. Evaluate  f. Plot loss & accuracy

## a. Import the necessary packages

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

I0000 00:00:1793641167.534999   40560 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793641169.308503   40560 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## b. Load the training and testing data (CIFAR-10)

In [2]:
# Path of dataset folder (change if your folder is somewhere else)
train_dir = "dataset/cifar-10-img/train"
test_dir  = "dataset/cifar-10-img/test"

datagen = ImageDataGenerator(rescale=1.0/255)   # pixel values 0-255 -> 0-1

train_gen = datagen.flow_from_directory(train_dir, target_size=(32, 32),
                                        color_mode="rgb", batch_size=50000,
                                        class_mode="categorical", shuffle=True)
test_gen  = datagen.flow_from_directory(test_dir, target_size=(32, 32),
                                        color_mode="rgb", batch_size=10000,
                                        class_mode="categorical", shuffle=True)

# One big batch = whole dataset loaded as numpy arrays
x_train, y_train = train_gen[0]
x_test,  y_test  = test_gen[0]
class_names = list(train_gen.class_indices.keys())

print("x_train:", x_train.shape, " y_train:", y_train.shape)
print("x_test :", x_test.shape,  " y_test :", y_test.shape)
print("Classes:", class_names)

Found 40079 images belonging to 10 classes.


Found 9921 images belonging to 10 classes.


x_train: (40079, 32, 32, 3)  y_train: (40079, 10)
x_test : (9921, 32, 32, 3)  y_test : (9921, 10)
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[np.argmax(y_train[i])])
    plt.axis("off")
plt.show()

/tmp/ipykernel_40560/676615008.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## c. Define the network architecture
Flatten (32x32x3 = 3072 inputs) → Dense(512, ReLU) → Dense(256, ReLU) → Dense(10, Softmax)

In [4]:
model = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Flatten(),
    layers.Dense(512, activation="relu"),
    layers.Dense(256, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,573,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,707,274 (6.51 MB)

 Trainable params: 1,707,274 (6.51 MB)

 Non-trainable params: 0 (0.00 B)

## d. Train the model using SGD

In [5]:
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
              loss="categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(x_train, y_train,
                    validation_data=(x_test, y_test),
                    epochs=20, batch_size=64)

Epoch 1/20


  1/627 ━━━━━━━━━━━━━━━━━━━━ 5:36 538ms/step - accuracy: 0.0469 - loss: 2.5341

  7/627 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.1362 - loss: 2.3128   

 13/627 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.1647 - loss: 2.2512 

 18/627 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1710 - loss: 2.2319

 24/627 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1732 - loss: 2.2222

... (1,881 outputs left out)

608/627 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6045 - loss: 1.1058

615/627 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6042 - loss: 1.1063

622/627 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.6043 - loss: 1.1057

627/627 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.6046 - loss: 1.1050 - val_accuracy: 0.4725 - val_loss: 1.6260


## e. Evaluate the network

In [6]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc*100:.2f}%")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d",
            xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

Test loss: 1.6260
Test accuracy: 47.25%


              precision    recall  f1-score   support

    airplane       0.59      0.48      0.53       986
  automobile       0.64      0.57      0.60      1057
        bird       0.37      0.30      0.33       982
         cat       0.30      0.51      0.38       974
        deer       0.48      0.36      0.41      1001
         dog       0.36      0.28      0.32       975
        frog       0.50      0.53      0.51      1013
       horse       0.53      0.56      0.54       975
        ship       0.64      0.51      0.57       955
       truck       0.47      0.62      0.53      1003

    accuracy                           0.47      9921
   macro avg       0.49      0.47      0.47      9921
weighted avg       0.49      0.47      0.47      9921



/tmp/ipykernel_40560/4274663811.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
n = 25   # change index to test another image
plt.imshow(x_test[n]); plt.axis("off"); plt.show()
pred = model.predict(x_test[n:n+1], verbose=0)
print("Predicted:", class_names[np.argmax(pred)])
print("Actual   :", class_names[np.argmax(y_test[n])])

Predicted: deer
Actual   : ship


/tmp/ipykernel_40560/714987791.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.imshow(x_test[n]); plt.axis("off"); plt.show()


## f. Plot the training loss and accuracy

In [8]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.show()

/tmp/ipykernel_40560/1834603888.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
